# Introduction to Big Data Tools in R

## What you'll learn
- What "big data" means in practice
- Why standard R tools struggle with large datasets
- Three tools that solve this: sparklyr, arrow, and duckdb
- How to choose the right tool for your situation

## Prerequisites
- Completed Notebook 01 (R Fundamentals)
- Familiarity with data frames and dplyr verbs

## What Is "Big Data"?

"Big data" doesn't have a single precise definition. In practice, it refers to data that is too large or complex to handle comfortably with standard tools on a single computer.

Some rough thresholds:

| Size | What it feels like in R |
|------|------------------------|
| < 100 MB | Fast. No special tools needed. |
| 100 MB – 1 GB | Starts to slow down. Reads take seconds, operations take longer. |
| 1 GB – 10 GB | Painful. May not fit in memory. R might crash. |
| 10 GB+ | Impossible with base R. You need specialized tools. |

Your computer has a limited amount of **RAM** (memory). When R loads a CSV file, it puts the entire thing into RAM. If the file is larger than your available RAM, R will either crash or grind to a halt.

Big data tools solve this by being smarter about how they read, store, and process data.

## Why Can't Base R Handle It?

Base R and the tidyverse are designed for **in-memory** processing. When you run `read_csv()`, the entire file is loaded into your computer's RAM. When you run `filter()` or `summarize()`, R creates copies and intermediate results — all in RAM.

This works great for datasets that fit comfortably in memory. But when data grows beyond that, you hit a wall.

The three tools we'll learn take different approaches to this problem:
1. **sparklyr** — distributes the work across multiple machines (or multiple cores)
2. **arrow** — uses a highly efficient in-memory format that can process data in chunks
3. **duckdb** — runs fast SQL queries directly on files, without loading everything into R

## The Three Tools

### sparklyr (Apache Spark for R)

**What it is:** An R interface to Apache Spark, a powerful engine designed to process massive datasets across clusters of computers.

**The key idea:** You write familiar dplyr code (`filter()`, `group_by()`, `summarize()`), but behind the scenes, Spark distributes the work across multiple cores or machines. You don't need to change how you think — just how the work gets done.

**Best for:** Very large datasets (tens of GB or more), especially when you have access to a computing cluster. Also useful locally for datasets that are too big for base R.

**Trade-off:** Requires Java and Spark to be installed (more setup). Has overhead — for small data, it's actually slower than base R.

### arrow (Apache Arrow for R)

**What it is:** An R package that uses the Apache Arrow format — a modern, highly efficient way to store and process columnar data.

**The key idea:** Arrow can read and process data without loading everything into R's memory at once. It's also much faster at reading files than `read_csv()`. You still use dplyr verbs — the syntax looks the same.

**Best for:** Speeding up file I/O, working with Parquet files (a modern alternative to CSV), and processing data that's a bit too big for comfortable base R use on a single machine.

**Trade-off:** Lightweight and fast, but it's designed for a single machine. Not a replacement for Spark on truly massive distributed datasets.

### duckdb (DuckDB for R)

**What it is:** An embedded analytical database that runs inside R. Think of it as a fast SQL engine that works directly on your files — no server needed.

**The key idea:** You write SQL queries (or use dplyr through `dbplyr`), and DuckDB executes them using very efficient algorithms. It can query CSV and Parquet files directly without loading them into memory first.

**Best for:** Fast analytical queries on local files. Excellent when you know SQL or want to mix SQL and R.

**Trade-off:** Single-machine only (like arrow). If your data is truly distributed across a cluster, you need Spark.

## Comparison Table

| Feature | sparklyr | arrow | duckdb |
|---------|----------|-------|--------|
| **Best for** | Distributed clusters, very large data | Fast file I/O, larger-than-memory files | Fast SQL analytics on local files |
| **Interface** | dplyr verbs on Spark | dplyr verbs on Arrow | SQL or dplyr (via dbplyr) |
| **Scales to** | Multiple machines (clusters) | Single machine, large files | Single machine |
| **Setup complexity** | High (needs Java + Spark) | Low (just install the package) | Low (just install the package) |
| **Reads CSV directly?** | Yes | Yes (very fast) | Yes (very fast) |
| **Parquet support?** | Yes | Yes (native) | Yes |
| **Learning curve** | Low if you know dplyr | Low if you know dplyr | Medium (SQL helps) |
| **Overhead for small data** | High (Spark startup) | Very low | Very low |

## When to Use Each Tool

Here's a simple decision guide:

1. **Your data fits in memory and operations are fast** → stick with tidyverse (dplyr + readr). No need for big data tools.

2. **Your data fits in memory but reads are slow, or you want Parquet support** → use **arrow**. It's the lightest upgrade.

3. **You want fast SQL-style analytics on local files** → use **duckdb**. It's fast and requires minimal setup.

4. **Your data is too large for one machine, or you're working on a cluster** → use **sparklyr**. It's the heavyweight option.

In practice, many data scientists use a combination. For example, you might use arrow to read Parquet files quickly, duckdb for fast aggregations, and sparklyr when you need to scale out to a cluster.

## A Quick Demo

Let's see how reading speed differs between base R and arrow. First, we need a larger dataset. Make sure you've generated it by running this command in your terminal:

```bash
python scripts/generate_large_data.py
```

This creates `data/sales_large.csv` with 100,000 rows.

In [ ]:
library(tidyverse)
library(arrow)

In [ ]:
# Time how long it takes to read with readr (tidyverse)
system.time({
  df_readr <- read_csv("../data/sales_large.csv", show_col_types = FALSE)
})

In [ ]:
# Time how long it takes to read with arrow
system.time({
  df_arrow <- read_csv_arrow("../data/sales_large.csv")
})

With 100K rows the difference may be small, but as files grow to millions of rows, arrow becomes dramatically faster.

We'll explore each tool in depth in the next three notebooks.

## What's Next

| Notebook | Tool | What you'll learn |
|----------|------|-------------------|
| 03 | sparklyr | Connect to Spark, run dplyr operations on Spark DataFrames |
| 04 | arrow | Fast file reading, Arrow tables, Parquet format |
| 05 | duckdb | SQL queries on files, DBI interface, dbplyr |

Each notebook is self-contained — you can do them in any order, or skip the ones that aren't relevant to your work. We recommend at least trying all three so you can make an informed choice.